# 05 — Pro Consumidor Prices | Automatic Download (raw landing)

Downloads the Pro Consumidor basic-goods price report published on the national open-data portal and lands it, unchanged, in the raw archive
`/Volumes/rd_economic_intelligence/bronze/proconsumidor_raw/consumer_prices/`.

| Step | Rule |
|---|---|
| Discover | datos.gob.do CKAN API, dataset `precios_productos_primera_necesidad`: current URL of the **XLSX** (ingested) and **CSV** (archived only; inconsistent delimiters) resources |
| Validate | The XLSX must contain the header `Id, Producto, Tipo, Precio, Mes, Año, establecimiento` in its first 10 rows. Otherwise the task fails and nothing is written. |
| Land | Shared helper `src/shared/acquisition.py`. Every quarterly report uses the **same file name**, so a new report is kept as `<name>__<date>` next to the archived ones; identical content is never landed twice. |
| Signal | Task value `downloaded` = true when at least one new file landed. |

Bronze (`05_ProConsumidor_Prices_Bronze`) then captures new workbooks incrementally with Auto Loader. **Status:** Development.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import sys, io, json, importlib
import pandas as pd

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import acquisition as ACQ
importlib.reload(ACQ)

LANDING = "/Volumes/rd_economic_intelligence/bronze/proconsumidor_raw/consumer_prices/"
PACKAGE = "precios_productos_primera_necesidad"
EXPECTED = ["Id", "Producto", "Tipo", "Precio", "Mes", "Año", "establecimiento"]

urls, portal_modified = ACQ.ckan_resources(PACKAGE, ["xlsx", "csv"])
print(f"Portal dataset updated {portal_modified}: {urls}")

files = {fmt: ACQ.http_get(url) for fmt, url in urls.items()}
head = pd.read_excel(io.BytesIO(files["xlsx"]), header=None, dtype=object, nrows=10)
if not any([str(v).strip() for v in head.iloc[i, :7]] == EXPECTED for i in range(len(head))):
    raise ValueError(f"XLSX failed the structural contract: header {EXPECTED} not found in the first 10 rows")

results = {fmt: ACQ.land(data, ACQ.urllib.parse.unquote(urls[fmt].rsplit("/", 1)[-1]), LANDING) for fmt, data in files.items()}
downloaded = any(r["status"] == "DOWNLOADED" for r in results.values())
dbutils.jobs.taskValues.set(key="downloaded", value=downloaded)
dbutils.notebook.exit(json.dumps({"status": "DOWNLOADED" if downloaded else "SKIPPED", "portal_modified": portal_modified, "files": results}, ensure_ascii=False))